In [4]:
from requests import request
from requests.compat import *
from bs4 import BeautifulSoup
import re
from requests.exceptions import HTTPError
import pandas as pd

In [10]:
headers = {'user-agent':'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'}

In [ ]:
start, end = map(int, input('몇페이지부터 몇페이지까지?: ').split())

for page in range(start, end):
    vurl ='https://ent.sbs.co.kr/news/flash.do?plink=GNB&cooper=SBSENTERNEWS&pageIdx={}'.format(page)
    urls = [vurl] # 앞으로 방문할곳
    visited = [] # 이미 방문한곳
    while urls: # 더이상 방문할 곳이 없을때까지
        url = urls.pop(0) # 앞에서부터 하나씩 꺼내고(FIFO)

        # HTTP 받아오고
        resp = request('GET', url, headers=headers)

        # 잘 받았으면

        visited.append(url)

            # Scarping(대상: text/html)
        if re.search('html', resp.headers['content-type']):
            # DOM 만들어주고
            dom = BeautifulSoup(resp.text, 'lxml')

            # 전략3-1. 특정 영역 - 뉴스 목록(주소만)
            # pagelink는 제외시킴
            news = dom.select('.news_list [href]')

            # 링크 찾기
            if len(news) > 0:
                a = []
                for temp in news:

                    temp = temp.attrs['href']
                    newurl = urljoin(url, temp)

                    a.append(newurl)


                    urls += list(filter(
                    lambda link:link not in visited and link not in urls, a))
                    
            # 전략3-2. 뉴스 본문만
            news = dom.select_one('.w_ctma_text')
            print(news)


            if news:
                # https://ent.sbs.co.kr/news/article.do?article_id=E10010275065
                with open(url.split('/')[-1].replace('?', '.')+'.txt',
                          'w', encoding='utf8') as fp:
                    fp.write(news.text)
        # 못받았으면

            # 방문했다고 치고, 방문안하게
        else:
            visited.append(url)

In [ ]:
url = 'https://www.hankookilbo.com/News/Enter'